
**Paper**: Using neural networks as an alternative to air dispersion modeling in environmental impact assessment.

**Authors**: Mateo Concha and Gonzalo Ruz.

**Description**: Code used for the paper development.



In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# --- Load datasets ---
train_data     = pd.read_csv("train.csv")         # 6000 rows
validate_data  = pd.read_csv("validation.csv")    # 1000 rows
test_FFEE_data = pd.read_csv("test_FFEE.csv")     # 1750 rows
test_CV_data   = pd.read_csv("test_CV.csv")       # 2947 rows


print("(shape - train)", train_data.shape)
print("(shape - validate)", validate_data.shape)
print("(shape - test_FFEE)", test_FFEE_data.shape)
print("(shape - test_CV)", test_CV_data.shape)

target_ffee = "FFEE"
target_cv   = "Chacaya"


feature_cols = [c for c in train_data.columns if c != target_ffee]

X_train = train_data[feature_cols]
y_train = train_data[target_ffee]

X_val = validate_data[feature_cols]
y_val = validate_data[target_ffee]

X_test_FFEE = test_FFEE_data[feature_cols]
y_test_FFEE = test_FFEE_data[target_ffee]


X_test_CV = test_CV_data.drop(columns=[target_cv]).reindex(columns=feature_cols)
y_test_CV = test_CV_data[target_cv]

# --- Scaling: fit on TRAIN only ---
scaler = MinMaxScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
X_test_FFEE = scaler.transform(X_test_FFEE)
X_test_CV   = scaler.transform(X_test_CV)


In [ ]:
pip install tensorflow

In [ ]:
## Training, validation, and tests datasets
train_data = pd.read_csv('Dataset/train.csv')
validate_data = pd.read_csv('Dataset/Validation.csv')
test_FEEE_data = pd.read_csv("Dataset/test_FFEE.csv")
test_CV_data = pd.read_csv("Dataset/test_CV.csv")

train_data, _ = train_test_split(train_data,
                                 test_size=0.3,
                                 random_state=123)
print('(shape - train) {}'.format(train_data.shape))
print('(shape - validate) {}'.format(validate_data.shape))
print('(shape - test_FFEE) {}'.format(test_FEEE_data.shape))
print('(shape - test_CV) {}'.format(test_CV_data.shape))

## Separate the datasets into predictors and targets (SO2 Concentration)
X_train, y_train = train_data.drop('FFEE', axis=1), train_data['FFEE']
X_val, y_val = validate_data.drop('FFEE', axis=1), validate_data['FFEE']
X_test_FFEE, y_test_FFEE = test_FEEE_data.drop('FFEE', axis=1), test_FEEE_data['FFEE']
X_test_CV, y_test_CV = test_CV_data.drop('Chacaya', axis=1), test_CV_data['Chacaya']

## Data Scaling
scaler = MinMaxScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test_FFEE = scaler.transform(X_test_FFEE)
X_test_CV = scaler.transform(X_test_CV)

In [ ]:
# NN random search biased toward generalization (smaller + more regularized)
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rng = np.random.default_rng(123)
tf.random.set_seed(123)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def build_mlp(input_dim, n_layers, units, activation, dropout, l2, lr):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for _ in range(n_layers):
        model.add(layers.Dense(
            units,
            activation=activation,
            kernel_regularizer=regularizers.l2(l2)
        ))
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation="linear"))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="mse"
    )
    return model

def log_uniform(low, high):
    return float(np.exp(rng.uniform(np.log(low), np.log(high))))

def sample_config():
    n_layers = int(rng.integers(1, 4))                    
    units = int(rng.choice([16, 32, 48, 64]))             
    activation = rng.choice(["relu", "elu"])             
    dropout = float(rng.choice([0.10, 0.15, 0.20, 0.30])) 
    l2 = float(rng.choice([1e-4, 3e-4, 1e-3]))            
    lr = log_uniform(1e-4, 1e-3)                          
    batch_size = int(rng.choice([64, 128, 256]))
    return dict(n_layers=n_layers, units=units, activation=activation,
                dropout=dropout, l2=l2, lr=lr, batch_size=batch_size)

TRIALS = 60  

early = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=25, restore_best_weights=True
)

best = {"val_rmse": np.inf, "val_mae": np.inf, "cfg": None}

for t in range(TRIALS):
    cfg = sample_config()
    model = build_mlp(
        X_train.shape[1],
        n_layers=cfg["n_layers"],
        units=cfg["units"],
        activation=cfg["activation"],
        dropout=cfg["dropout"],
        l2=cfg["l2"],
        lr=cfg["lr"]
    )
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=600,
        batch_size=cfg["batch_size"],
        verbose=0,
        callbacks=[early]
    )
    pred_val = model.predict(X_val, verbose=0).flatten()
    v_rmse = rmse(y_val, pred_val)
    v_mae = float(mean_absolute_error(y_val, pred_val))

    if v_rmse < best["val_rmse"]:
        best.update({"val_rmse": v_rmse, "val_mae": v_mae, "cfg": cfg})

print("Best NN config (generalization-biased search):", best["cfg"])
print(f"Best validation: RMSE={best['val_rmse']:.3f} | MAE={best['val_mae']:.3f}")

# Refit on train+val and evaluate tests
X_tr_all = np.vstack([X_train, X_val])
y_tr_all = np.concatenate([y_train, y_val])

final = build_mlp(
    X_train.shape[1],
    n_layers=best["cfg"]["n_layers"],
    units=best["cfg"]["units"],
    activation=best["cfg"]["activation"],
    dropout=best["cfg"]["dropout"],
    l2=best["cfg"]["l2"],
    lr=best["cfg"]["lr"]
)

final.fit(
    X_tr_all, y_tr_all,
    epochs=600,
    batch_size=best["cfg"]["batch_size"],
    verbose=0,
    callbacks=[early]
)

pred_f = final.predict(X_test_FFEE, verbose=0).flatten()
pred_cv = final.predict(X_test_CV, verbose=0).flatten()

print(f"Test FFEE: RMSE={rmse(y_test_FFEE, pred_f):.3f} | MAE={mean_absolute_error(y_test_FFEE, pred_f):.3f} | R^2={r2_score(y_test_FFEE, pred_f):.3f}")
print(f"Test CV:   RMSE={rmse(y_test_CV, pred_cv):.3f} | MAE={mean_absolute_error(y_test_CV, pred_cv):.3f} | R^2={r2_score(y_test_CV, pred_cv):.3f}")
